In [0]:
# Table: gold_daily_weather_traffic_correlation
# Description: Correlates daily weather and traffic data by zone

from pyspark.sql.functions import sum, avg, to_date, col, round

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")
weather_df = spark.table("sagar_cap3_cat1.silver.weather_all_zones")

weather_traffic_correlation = (
    stream_df.join(weather_df, on="zone", how="left")
    .where(to_date(col("event_ts")) == to_date(col("timestamp")))
    .groupBy(
        to_date(col("event_ts")).alias("report_date"),
        col("zone")
    )
    .agg(
        round(avg("avg_speed_kmh"), 2).alias("avg_speed_kph"),
        sum("vehicle_count").alias("daily_vehicle_count"),
        round(avg("aqi"), 2).alias("avg_aqi"),
        round(avg("precip_mm"), 2).alias("avg_precipitation_mm"),
        round(avg("temperature_c"), 2).alias("avg_temperature_c")
    )
    .orderBy("report_date", "zone")
)

weather_traffic_correlation.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.gold_daily_weather_traffic_correlation")
display(weather_traffic_correlation)

report_date,zone,avg_speed_kph,daily_vehicle_count,avg_aqi,avg_precipitation_mm,avg_temperature_c
2025-08-29,CENTRAL,53.28,138137544,47.7,2.58,24.12
2025-08-29,NORTH,52.69,101400168,47.77,2.74,27.42
2025-08-29,NORTHWEST,53.7,131561184,47.59,2.49,26.02
2025-08-29,SOUTH,55.55,43323120,46.53,1.99,26.7
2025-08-29,WEST,51.75,33909744,48.27,2.98,24.5


In [0]:
### intersection_aqi_summary
from pyspark.sql.functions import avg, sum, to_date, col

intersection_aqi_summary = (
    spark.table("sagar_cap3_cat1.silver.events_data")
    .groupBy(
        to_date("event_ts").alias("report_date"),
        col("intersection_id")
    )
    .agg(
        round(avg("aqi"), 2).alias("daily_avg_aqi"), 
        sum("vehicle_count").alias("daily_vehicle_count"),
        round(avg("avg_speed_kmh"), 2).alias("daily_avg_speed_kph") 
    )
    .orderBy("report_date", "intersection_id")
)

intersection_aqi_summary.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.intersection_aqi_summary")
display(intersection_aqi_summary)

In [0]:
#Table name: gold_intraday_traffic_summary
## Traffic Volume Comparison: Holidays vs. Normal Days
from pyspark.sql.functions import avg, sum, to_date, when, col, hour

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")
holidays_df = spark.table("sagar_cap3_cat1.silver.holiday_calendar")

holiday_comparison_agg = (
    stream_df.join(holidays_df, to_date(col("event_ts")) == col("date"), how="left")
    .groupBy(
        when(col("holiday_name").isNotNull(), "Holiday")
        .otherwise("Normal Day").alias("day_type"),
        hour("event_ts").alias("hour_of_day")
    )
    .agg(
        avg("avg_speed_kmh").alias("avg_speed_kph"),
        sum("vehicle_count").alias("total_vehicles")
    )
    .orderBy("day_type", "hour_of_day")
)

from pyspark.sql.functions import sum, avg, window, col

intraday_traffic_summary = (
    stream_df.groupBy(
        window("event_ts", "15 minutes").alias("window"),
        "intersection_id",
        "corridor_id",
        "zone"
    )
    .agg(
        sum("vehicle_count").alias("15_min_vehicle_count"),
        avg("avg_speed_kmh").alias("15_min_avg_speed_kph"),
        avg("aqi").alias("15_min_avg_aqi")
    )
    .orderBy("window", "intersection_id")
)
intraday_traffic_summary.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.gold_intraday_traffic_summary")
display(intraday_traffic_summary)

window,intersection_id,corridor_id,zone,15_min_vehicle_count,15_min_avg_speed_kph,15_min_avg_aqi
"List(2025-08-29T00:45:00.000Z, 2025-08-29T01:00:00.000Z)",X00897,C001,NORTHWEST,4,53.59,43.0
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00002,C007,NORTH,121,64.96633333333334,41.63333333333333
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00003,C002,NORTHWEST,58,49.5225,42.95
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00004,C001,NORTHWEST,183,63.14216666666665,41.46666666666667
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00006,C004,NORTH,133,65.095,41.53333333333333
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00012,C007,NORTH,129,61.21166666666665,41.61666666666667
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00014,C003,SOUTH,51,49.65866666666667,46.63333333333333
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00018,C005,CENTRAL,81,64.37666666666668,44.06666666666667
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00019,C005,CENTRAL,297,63.14944444444444,42.13333333333333
"List(2025-08-29T01:00:00.000Z, 2025-08-29T01:15:00.000Z)",X00020,C001,NORTHWEST,207,63.54616666666667,40.86666666666667


In [0]:
### gold_corridor_traffic_by_hour

from pyspark.sql.functions import sum, avg, hour, col

hourly_corridor_traffic = (
    spark.table("sagar_cap3_cat1.silver.events_data")
    .groupBy(
        col("corridor_id"),
        hour("event_ts").alias("hour_of_day")
    )
    .agg(
        sum("vehicle_count").alias("total_hourly_vehicles"),
        round(avg("avg_speed_kmh"),2).alias("avg_hourly_speed_kph")  # m2
    )
    .orderBy("corridor_id", "hour_of_day")
)

hourly_corridor_traffic.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.corridor_traffic_by_hour")
display(hourly_corridor_traffic)

corridor_id,hour_of_day,total_hourly_vehicles,avg_hourly_speed_kph
C001,0,4,53.59
C001,1,31630,63.98
C001,2,26746,63.61
C001,3,36792,45.87
C001,4,31720,57.55
C001,5,72457,63.97
C001,6,98649,50.65
C001,7,113184,42.67
C001,8,134310,47.07
C001,9,144645,54.9


In [0]:
# Table: critical_location_impact
# Description: Analyzes traffic impact near critical locations like schools and hospitals

from pyspark.sql.functions import sum, avg, hour, col, when, to_date

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")
intersections_df = spark.table("sagar_cap3_cat1.silver.intersections")

critical_location_impact = (
    stream_df.join(intersections_df, on="intersection_id")
    .where((col("near_school") == True) | (col("near_hospital") == True))
    .groupBy(
        col("intersection_id"),
        to_date("event_ts").alias("report_date"),
        hour("event_ts").alias("hour_of_day"),
        when(col("near_school") == True, "School").otherwise(None).alias("near_school"),
        when(col("near_hospital") == True, "Hospital").otherwise(None).alias("near_hospital")
    )
    .agg(
        avg("avg_speed_kmh").alias("avg_speed_kph"),
        sum("vehicle_count").alias("total_vehicles"),
        avg("aqi").alias("avg_aqi_near_location")
    )
    .orderBy("report_date", "hour_of_day")
)

critical_location_impact.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.critical_location_impact")
display(critical_location_impact)

intersection_id,report_date,hour_of_day,near_school,near_hospital,avg_speed_kph,total_vehicles,avg_aqi_near_location
X00897,2025-08-29,0,School,null,53.59,4,43.0
X00740,2025-08-29,1,School,null,63.830916666666624,456,43.50833333333333
X00018,2025-08-29,1,School,null,63.24416666666668,324,43.9
X00671,2025-08-29,1,School,null,64.26416666666665,486,44.43333333333333
X00418,2025-08-29,1,null,Hospital,64.19358333333332,218,41.041666666666664
X00585,2025-08-29,1,null,Hospital,63.78049999999998,5219,42.69166666666667
X00733,2025-08-29,1,School,null,64.51116666666665,213,43.175
X00332,2025-08-29,1,School,null,55.592250000000014,220,46.266666666666666
X00818,2025-08-29,1,School,null,63.22749999999999,486,43.458333333333336
X00196,2025-08-29,1,null,Hospital,62.646750000000004,480,41.59166666666667


In [0]:
## Table: critical_locations_agg
## Hourly Congestion Near Critical Locations (Schools/Hospitals)
from pyspark.sql.functions import avg, sum, hour, when, col

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")
intersections_df = spark.table("sagar_cap3_cat1.silver.intersections")

critical_locations_agg = (
    stream_df.join(intersections_df, on="intersection_id")
    .where((col("near_school") == True) | (col("near_hospital") == True))
    .groupBy(
        col("intersection_id"),
        hour("event_ts").alias("hour_of_day"),
        when(col("near_school") == True, "School").otherwise(None).alias("near_school"),
        when(col("near_hospital") == True, "Hospital").otherwise(None).alias("near_hospital")
    )
    .agg(
        avg("avg_speed_kmh").alias("avg_speed_kph"),
        sum("vehicle_count").alias("total_vehicles"),
        avg("aqi").alias("avg_aqi_near_location")
    )
    .orderBy("hour_of_day", "avg_aqi_near_location", ascending=False)
)
critical_locations_agg.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.gold.critical_locations_agg')
display(critical_locations_agg)

intersection_id,hour_of_day,near_school,near_hospital,avg_speed_kph,total_vehicles,avg_aqi_near_location
X00066,23,School,null,41.41116666666669,5467,56.86666666666667
X00818,23,School,null,40.09525000000001,464,56.19166666666667
X00170,23,School,null,40.15816666666669,448,55.34166666666667
X00507,23,School,null,41.02516666666668,481,55.266666666666666
X00560,23,School,null,41.20166666666666,949,55.09166666666667
X00437,23,School,null,41.01737499999999,669,55.079166666666666
X00224,23,School,null,41.01691666666667,474,54.983333333333334
X00138,23,School,null,40.90291666666667,663,54.8375
X00654,23,School,null,42.34720833333333,676,54.8
X00423,23,School,null,41.38924999999999,306,54.541666666666664


In [0]:
# Table: holiday_traffic_impact
# Description: Analyzes traffic impact on holidays vs. normal days

from pyspark.sql.functions import avg, sum, to_date, when, col, hour

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")
holidays_df = spark.table("sagar_cap3_cat1.silver.holiday_calendar")

holiday_comparison_agg = (
    stream_df.join(holidays_df, to_date(col("event_ts")) == col("date"), how="left")
    .groupBy(
        when(col("holiday_name").isNotNull(), "Holiday").otherwise("Normal Day").alias("day_type"),
        hour("event_ts").alias("hour_of_day")
    )
    .agg(
        round(avg("avg_speed_kmh"),2).alias("avg_speed_kph"),
        sum("vehicle_count").alias("total_vehicles")
    )
    .orderBy("day_type", "hour_of_day")
)

holiday_comparison_agg.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.holiday_traffic_impact")
display(holiday_comparison_agg)

day_type,hour_of_day,avg_speed_kph,total_vehicles
Normal Day,0,53.59,4
Normal Day,1,61.54,332397
Normal Day,2,58.24,342249
Normal Day,3,53.25,313027
Normal Day,4,55.43,337125
Normal Day,5,57.34,609651
Normal Day,6,52.08,850365
Normal Day,7,51.32,899135
Normal Day,8,52.31,1277574
Normal Day,9,51.89,1268684


In [0]:
# Table: incidents_summary
# Description: Summarizes traffic incidents based on speed and sensor status

from pyspark.sql.functions import when, lit, col, lag, unix_timestamp
from pyspark.sql.window import Window

stream_df = spark.table("sagar_cap3_cat1.silver.events_data")

window_spec = Window.partitionBy("sensor_id").orderBy("event_ts")
incidents = (
    stream_df
    .withColumn("prev_speed", lag("avg_speed_kmh", 1).over(window_spec))
    .withColumn(
        "time_diff_sec", 
        unix_timestamp("event_ts") - unix_timestamp(lag("event_ts", 1).over(window_spec))
    )
    .withColumn("speed_drop", col("prev_speed") - col("avg_speed_kmh"))
    .withColumn(
        "incident_type", 
        when((col("time_diff_sec") < 60) & (col("speed_drop") > 40), "Abrupt Speed Collapse")
        .when(col("lane_status") == "offline", "Sensor Offline")
        .otherwise(lit("None"))
    )
    .filter(col("incident_type") != "None")
    .select(
        col("event_ts").alias("timestamp"),
        col("incident_type"),
        col("aqi"),
        col("avg_speed_kmh"),
        col("geo"),
        lit("1.0").alias("severity_score") 
    )
)

incidents.write.mode("append").saveAsTable("sagar_cap3_cat1.gold.incidents_summary")
display(incidents)

timestamp,incident_type,aqi,avg_speed_kmh,geo,severity_score
2025-08-29T01:12:00.000Z,Abrupt Speed Collapse,73,9.34,"List(28.75, 77.43)",1.0
2025-08-29T02:40:00.000Z,Abrupt Speed Collapse,35,62.94,"List(28.75, 77.43)",1.0
2025-08-29T05:34:30.000Z,Abrupt Speed Collapse,69,9.59,"List(28.75, 77.43)",1.0
2025-08-29T17:24:00.000Z,Abrupt Speed Collapse,74,4.44,"List(28.75, 77.43)",1.0
2025-08-29T18:49:00.000Z,Abrupt Speed Collapse,44,55.78,"List(28.75, 77.43)",1.0
2025-08-29T22:20:00.000Z,Abrupt Speed Collapse,75,0.0,"List(28.75, 77.43)",1.0
2025-08-29T03:56:30.000Z,Abrupt Speed Collapse,75,9.36,"List(28.42, 77.04)",1.0
2025-08-29T04:25:30.000Z,Abrupt Speed Collapse,42,49.0,"List(28.42, 77.04)",1.0
2025-08-29T08:32:30.000Z,Abrupt Speed Collapse,71,18.41,"List(28.42, 77.04)",1.0
2025-08-29T09:05:00.000Z,Abrupt Speed Collapse,73,6.58,"List(28.42, 77.04)",1.0


In [0]:
# Table: corridor_equipment_health_report
# Description: Aggregates health metrics of cameras and sensors by corridor

from pyspark.sql.functions import avg, count, countDistinct, col

cameras_df = spark.table("sagar_cap3_cat1.silver.cameras")
sensors_df = spark.table("sagar_cap3_cat1.silver.sensors")

camera_health = (
    cameras_df.groupBy("corridor_id")
    .agg(
        round(avg("uptime_pct"),2).alias("avg_camera_uptime_pct"),
        countDistinct("fault_codes").alias("num_camera_faults")
    )
)

sensor_health = (
    sensors_df.groupBy("corridor_id")
    .agg(
        count("sensor_id").alias("total_sensors"),
        countDistinct("firmware").alias("num_firmware_versions")
    )
)

equipment_health_report = (
    camera_health.join(sensor_health, on="corridor_id", how="outer")
)

equipment_health_report.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.corridor_equipment_health_report")
display(equipment_health_report)

corridor_id,avg_camera_uptime_pct,num_camera_faults,total_sensors,num_firmware_versions
C008,98.21,5,50,47
C002,97.57,4,52,49
C005,98.22,4,53,49
C001,98.6,5,66,58
C010,97.36,6,55,46
C007,98.51,5,76,63
C004,98.32,6,60,52
C003,97.52,3,59,53
C009,97.83,9,82,68
C006,97.82,6,47,45


In [0]:

### data_quality_report
from pyspark.sql.functions import count, countDistinct, to_date, col, lit

daily_data_quality = (
    spark.table("sagar_cap3_cat1.silver.events_data")
    .groupBy(to_date("event_ts").alias("report_date"))
    .agg(
        count("*").alias("total_events"),
        countDistinct("sensor_id").alias("unique_sensors"),
        sum(when(col("vehicle_count").isNull(), 1).otherwise(0)).alias("null_vehicle_counts"),
        sum(when(col("avg_speed_kmh").isNull(), 1).otherwise(0)).alias("null_speed_counts"),
        sum(when(col("aqi").isNull(), 1).otherwise(0)).alias("null_aqi_counts")
    )
    .withColumn("data_completeness_pct", (col("total_events") - col("null_vehicle_counts") - col("null_speed_counts") - col("null_aqi_counts")) / col("total_events") * 100)
    .orderBy("report_date")
)

daily_data_quality.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.gold.data_quality_report")
display(daily_data_quality)


report_date,total_events,unique_sensors,null_vehicle_counts,null_speed_counts,null_aqi_counts,data_completeness_pct
2025-08-29,1655000,600,0,0,0,100.0
